In [2]:
# @title {"single-column":true,"display-mode":"form"}

DATASET = "Cancer" # @param ["Cancer", "Somatic + Germline"]

OUTPUT_TYPE = "Proportions" # @param ["Counts", "Proportions"]

MIN_COUNT = 5 # @param {type:"integer"}

In [3]:
import pandas as pd
import re
import httpimport

with httpimport.github_repo("glygener", "colab-notebooks", ref="main"):
    from glygen import GlyGenDownloader

ggdl = GlyGenDownloader()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
template = "{species}_protein_mutation_germline_glycoeffect.csv"
germ_files = ggdl.filenames(template, species="human")

template = "{species}_protein_mutation_somatic_glycoeffect.csv"
som_files = ggdl.filenames(template, species="human")

template = "{species}_protein_mutation_cancer_glycoeffect.csv"
cancer_files = ggdl.filenames(template, species="human")


params = {
    "usecols": [
        "uniprotkb_canonical_ac",
        "aa_pos",
        "ref_aa",
        "alt_aa",
        "do_id",
        "effect"
    ],
    "dropdups": True,
    "notna": ["do_id"]
}

germ = ggdl.dataframe(germ_files, **params)
som = ggdl.dataframe(som_files, **params)
cancer = ggdl.dataframe(cancer_files, **params)

Download human_protein_mutation_germline_glycoeffect.csv...
Download progress: 269MB [00:01, 168MB/s]                           
Download human_protein_mutation_germline_glycoeffect.csv... done (268.91 MB).
Constructed DataFrame:

<class 'pandas.core.frame.DataFrame'>
Index: 13797 entries, 0 to 16388
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   uniprotkb_canonical_ac  13797 non-null  object 
 1   aa_pos                  13797 non-null  int64  
 2   ref_aa                  13797 non-null  object 
 3   alt_aa                  12631 non-null  object 
 4   do_id                   13797 non-null  float64
 5   effect                  13797 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 754.5+ KB

Download human_protein_mutation_somatic_glycoeffect.csv...
Download progress: 104MB [00:00, 118MB/s]                           
Download human_protein_mutation_somatic_glycoeffe

In [ ]:
url = "https://raw.githubusercontent.com/<org>/<repo>/main/data/HumanDO.tsv"

mapping_df = pd.read_csv(url, sep="\t")

In [ ]:
columns = [
    "uniprotkb_canonical_ac",
    "aa_pos",
    "ref_aa",
    "alt_aa",
    "do_id",
    "effect"
]

germ = germ[columns]
som = som[columns]
cancer = cancer[columns]

def clean_dataframe(df):
    df = df.dropna()
    df["do_id"] = df["do_id"].astype(int)
    return df.drop_duplicates()

cancer = clean_dataframe(cancer)

som_germ = pd.concat([som, germ], ignore_index=True)
som_germ = clean_dataframe(som_germ)

In [ ]:
mapping_df["id_int"] = (
    mapping_df["id"]
    .str.replace("DOID:", "", regex=False)
    .astype(int)
)

do_mapping = dict(
    zip(mapping_df["id_int"], mapping_df["subClassOf"])
)

cancer["do_name"] = cancer["do_id"].map(do_mapping)
som_germ["do_name"] = som_germ["do_id"].map(do_mapping)

In [ ]:
def categorize_cancer(text):
    """Categorize diseases into cancer types."""
    text = text.lower()

    # --- First: is it cancer at all? ---
    if not re.search(
        r"cancer|carcinoma|tumou?r|neoplasm|leukemia|lymphoma|sarcoma",
        text
    ):
        return "Other"

    # --- Nervous system ---
    if re.search(
        r"brain|central nervous system|cns|glioma|astrocytoma|infratentorial|retina|retinal",
        text
    ):
        return "Nervous System Cancer"

    # --- Hematologic ---
    if re.search(
        r"leukemia|lymphoma|myeloma|bone marrow|hematologic|blood cancer",
        text
    ):
        return "Hematologic Cancer"

    # --- Gastrointestinal ---
    if re.search(
        r"stomach|gastric|intestinal|colon|colorectal|large intestine|small intestine|gi|gastrointestinal",
        text
    ):
        return "Gastrointestinal Cancer"

    # --- Hepatobiliary ---
    if re.search(
        r"liver|hepatic|hepatobiliary|bile duct|cholangiocarcinoma",
        text
    ):
        return "Hepatobiliary Cancer"

    # --- Respiratory ---
    if re.search(
        r"lung|pulmonary|bronchial|respiratory",
        text
    ):
        return "Respiratory Cancer"

    # --- Endocrine ---
    if re.search(
        r"thyroid|adrenal|pituitary|endocrine|parathyroid",
        text
    ):
        return "Endocrine Cancer"

    # --- Skin ---
    if re.search(
        r"skin|melanoma|cutaneous|integumentary",
        text
    ):
        return "Skin Cancer"

    # --- Renal / Urinary ---
    if re.search(
        r"kidney|renal|bladder|urinary|urothelial",
        text
    ):
        return "Renal/Urinary Cancer"

    # --- Female reproductive ---
    if re.search(
        r"breast|ovarian|ovary|uterine|endometrial|cervical|female reproductive",
        text
    ):
        return "Female Reproductive Cancer"

    # --- Male reproductive ---
    if re.search(
        r"prostate|testicular|testis|male reproductive",
        text
    ):
        return "Male Reproductive Cancer"

    # --- Pancreatic ---
    if re.search(r"pancrea", text):
        return "Pancreatic Cancer"

    # --- Fallback ---
    return "Unspecified Cancer"


In [ ]:
def categorize_noncancer(text):
    """Categorize diseases into broad disease classes."""
    text = text.lower()

    if re.search(
        r"cancer|carcinoma|tumor|neoplasm|leukemia|lymphoma|sarcoma",
        text
    ):
        return "Cancer-Risk"

    elif re.search(
        r"neuro|alzheimer|parkinson|epilepsy|ataxia|dementia|sclerosis|migraine|cmt|charcot",
        text
    ):
        return "Neurological"

    elif re.search(
        r"cardio|heart|cardiomyopathy|arrhythmia|aortic|atrial|vascular|artery|hypertension|thromb|sinoatrial",
        text
    ):
        return "Cardiovascular"

    elif re.search(
        r"diabetes|metabolic|glycogen|lipid|cholesterol|endocrin|thyroid",
        text
    ):
        return "Metabolic"

    elif re.search(
        r"immune|autoimmune|inflammatory|immunodeficiency|lupus",
        text
    ):
        return "Immune"

    elif re.search(
        r"syndrome|congenital|developmental|intellectual|microcephaly|dysplasia|cranio|acro|syndactyly",
        text
    ):
        return "Developmental"

    elif re.search(
        r"muscular|dystrophy|myopathy|skeletal|bone|osteoporosis",
        text
    ):
        return "Musculoskeletal"

    elif re.search(
        r"kidney|renal|hypogonadism|spermatogenic|nephro",
        text
    ):
        return "Renal"

    elif re.search(
        r"deafness|hearing|macular|vision|retina|blindness",
        text
    ):
        return "Sensory"

    elif re.search(
        r"anemia|spherocytosis|hemoglob|sickle|protein s|protein c",
        text
    ):
        return "Hematological"

    elif re.search(
        r"hereditary|familial",
        text
    ):
        return "Hereditary"

    else:
        return "Uncategorized"

In [ ]:
if DATASET == "Cancer":
    df = cancer.copy()
    df["category"] = df["do_name"].apply(categorize_cancer)

else:
    df = som_germ.copy()
    df["category"] = df["do_name"].apply(categorize_noncancer)

In [ ]:
counts = (
    df.groupby(["effect", "category"])
      .size()
      .unstack(fill_value=0)
)

counts = counts.loc[counts.sum(axis=1) >= MIN_COUNT]

proportions = counts.div(counts.sum(axis=1), axis=0)

In [ ]:
if OUTPUT_TYPE == "Counts":
    result = counts
else:
    result = proportions

result